# Layer 1: Abbreviation → IANA Zone

Abbreviations are NOT unique. "IST" is India, Ireland, and Israel.
"EST" is US Eastern (UTC-5) AND Australian Eastern (UTC+10).

So this layer returns a LIST of candidates, not a single answer.

Sources:
1. Generate from the tz database — ask every zone what its abbreviation is.
2. Curate a small table of famous ones for ranking.

In [1]:
from datetime import datetime, timezone
import zoneinfo
from collections import defaultdict

now_utc = datetime.now(timezone.utc)

# abbrev -> list of zones that currently use it
abbrev_map = defaultdict(list)

for zone_name in zoneinfo.available_timezones():
    try:
        tz = zoneinfo.ZoneInfo(zone_name)
        local = now_utc.astimezone(tz)
        abbrev = local.tzname()
        if abbrev:
            abbrev_map[abbrev].append(zone_name)
    except Exception:
        continue

# How many distinct abbreviations are there?
print(f"distinct abbreviations: {len(abbrev_map)}")

# Look at the ones people actually ask about
for abbr in ["IST", "CST", "EST", "PST", "GMT", "BST", "JST", "AEST"]:
    zones = abbrev_map.get(abbr, [])
    print(f"\n{abbr} -> {len(zones)} zone(s)")
    for z in zones[:5]:
        print(f"    {z}")

distinct abbreviations: 80

IST -> 4 zone(s)
    Europe/Dublin
    Asia/Kolkata
    Eire
    Asia/Calcutta

CST -> 24 zone(s)
    Asia/Harbin
    America/Managua
    Asia/Chongqing
    America/Chihuahua
    America/Costa_Rica

EST -> 8 zone(s)
    America/Cancun
    America/Coral_Harbour
    America/Panama
    America/Atikokan
    America/Cayman

PST -> 1 zone(s)
    Asia/Manila

GMT -> 28 zone(s)
    GMT+0
    Etc/GMT
    Atlantic/Reykjavik
    Africa/Monrovia
    Africa/Ouagadougou

BST -> 7 zone(s)
    Europe/London
    Europe/Belfast
    Europe/Jersey
    Europe/Guernsey
    GB

JST -> 2 zone(s)
    Japan
    Asia/Tokyo

AEST -> 3 zone(s)
    Australia/Lindeman
    Australia/Brisbane
    Australia/Queensland


## What you'll notice

- Many abbreviations map to a single zone (JST, GMT, BST).
- Some map to many (IST probably maps to Asia/Kolkata, Europe/Dublin,
  Asia/Jerusalem, and possibly more).
- Some zones have numeric abbreviations like "+0545" (Nepal) or "-03"
  (various). These are NOT human-friendly — you'll want to filter them
  out of the "type an abbreviation" lookup.
- The set of zones for a given abbreviation can change over time
  (a zone might use "IST" in summer and "GMT" in winter).

In [2]:
# Only keep abbreviations that look human-readable.
# Heuristic: no digits, no plus/minus, no leading dash.
def is_human_abbrev(a):
    if not a:
        return False
    return a.isalpha() and a.isupper()

human_map = {
    abbr: zones for abbr, zones in abbrev_map.items()
    if is_human_abbrev(abbr)
}
print(f"human-readable abbreviations: {len(human_map)}")
for abbr in sorted(human_map):
    print(f"  {abbr:<6} -> {len(human_map[abbr])} zone(s)")

human-readable abbreviations: 43
  ACDT   -> 4 zone(s)
  ACST   -> 2 zone(s)
  ADT    -> 7 zone(s)
  AEDT   -> 10 zone(s)
  AEST   -> 3 zone(s)
  AKDT   -> 7 zone(s)
  AST    -> 24 zone(s)
  AWST   -> 2 zone(s)
  BST    -> 7 zone(s)
  CAT    -> 11 zone(s)
  CDT    -> 20 zone(s)
  CEST   -> 37 zone(s)
  CET    -> 2 zone(s)
  CST    -> 24 zone(s)
  EAT    -> 11 zone(s)
  EDT    -> 27 zone(s)
  EEST   -> 23 zone(s)
  EET    -> 3 zone(s)
  EST    -> 8 zone(s)
  GMT    -> 28 zone(s)
  HDT    -> 3 zone(s)
  HKT    -> 2 zone(s)
  HST    -> 4 zone(s)
  IDT    -> 3 zone(s)
  IST    -> 4 zone(s)
  JST    -> 2 zone(s)
  KST    -> 3 zone(s)
  MDT    -> 12 zone(s)
  MSK    -> 5 zone(s)
  MST    -> 12 zone(s)
  NDT    -> 2 zone(s)
  NZDT   -> 4 zone(s)
  PDT    -> 9 zone(s)
  PKT    -> 1 zone(s)
  PST    -> 1 zone(s)
  SAST   -> 3 zone(s)
  SST    -> 4 zone(s)
  UTC    -> 8 zone(s)
  WAT    -> 11 zone(s)
  WEST   -> 7 zone(s)
  WIB    -> 2 zone(s)
  WIT    -> 1 zone(s)
  WITA   -> 2 zone(s)


In [3]:
for abbr in ["IST", "CST", "EST", "AST", "BST"]:
    zones = human_map.get(abbr, [])
    print(f"\n{abbr} ({len(zones)} candidates):")
    for z in sorted(zones):
        print(f"    {z}")


IST (4 candidates):
    Asia/Calcutta
    Asia/Kolkata
    Eire
    Europe/Dublin

CST (24 candidates):
    America/Bahia_Banderas
    America/Belize
    America/Chihuahua
    America/Costa_Rica
    America/El_Salvador
    America/Guatemala
    America/Managua
    America/Merida
    America/Mexico_City
    America/Monterrey
    America/Regina
    America/Swift_Current
    America/Tegucigalpa
    Asia/Chongqing
    Asia/Chungking
    Asia/Harbin
    Asia/Macao
    Asia/Macau
    Asia/Shanghai
    Asia/Taipei
    Canada/Saskatchewan
    Mexico/General
    PRC
    ROC

EST (8 candidates):
    America/Atikokan
    America/Cancun
    America/Cayman
    America/Coral_Harbour
    America/Jamaica
    America/Panama
    EST
    Jamaica

AST (24 candidates):
    America/Anguilla
    America/Antigua
    America/Aruba
    America/Barbados
    America/Blanc-Sablon
    America/Curacao
    America/Dominica
    America/Grenada
    America/Guadeloupe
    America/Kralendijk
    America/Lower_Princes
  

## Design decision

For the World Clock, do we:

(a) Return a single best guess? Wrong often. Bad UX.
(b) Return a list of candidates? Correct. Requires a picker UI.
(c) Return a single candidate only when unambiguous, else a list?
    Best of both. "JST" -> one answer. "IST" -> three choices.

Option (c) is what production apps do. Implement that.

In [4]:
def resolve_abbreviation(abbr):
    """Return (status, candidates).

    status is one of:
      "unique"     - exactly one zone; candidates has one item
      "ambiguous"  - multiple zones; user must pick
      "unknown"    - no zone uses this abbreviation
    """
    a = abbr.strip().upper()
    zones = human_map.get(a, [])
    if not zones:
        return "unknown", []
    if len(zones) == 1:
        return "unique", zones
    return "ambiguous", sorted(zones)

In [5]:
for abbr in ["JST", "GMT", "IST", "CST", "XYZ", "est", "  ist  "]:
    status, zones = resolve_abbreviation(abbr)
    print(f"{abbr!r:<10} -> {status:<10} {zones}")

'JST'      -> ambiguous  ['Asia/Tokyo', 'Japan']
'GMT'      -> ambiguous  ['Africa/Abidjan', 'Africa/Accra', 'Africa/Bamako', 'Africa/Banjul', 'Africa/Bissau', 'Africa/Conakry', 'Africa/Dakar', 'Africa/Freetown', 'Africa/Lome', 'Africa/Monrovia', 'Africa/Nouakchott', 'Africa/Ouagadougou', 'Africa/Sao_Tome', 'Africa/Timbuktu', 'America/Danmarkshavn', 'Atlantic/Reykjavik', 'Atlantic/St_Helena', 'Etc/GMT', 'Etc/GMT+0', 'Etc/GMT-0', 'Etc/GMT0', 'Etc/Greenwich', 'GMT', 'GMT+0', 'GMT-0', 'GMT0', 'Greenwich', 'Iceland']
'IST'      -> ambiguous  ['Asia/Calcutta', 'Asia/Kolkata', 'Eire', 'Europe/Dublin']
'CST'      -> ambiguous  ['America/Bahia_Banderas', 'America/Belize', 'America/Chihuahua', 'America/Costa_Rica', 'America/El_Salvador', 'America/Guatemala', 'America/Managua', 'America/Merida', 'America/Mexico_City', 'America/Monterrey', 'America/Regina', 'America/Swift_Current', 'America/Tegucigalpa', 'Asia/Chongqing', 'Asia/Chungking', 'Asia/Harbin', 'Asia/Macao', 'Asia/Macau', 'Asia/Shanghai

In [6]:
def snapshot(zone_name):
    tz = zoneinfo.ZoneInfo(zone_name)
    local = datetime.now(timezone.utc).astimezone(tz)
    return local.strftime("%H:%M %Z (%z)")

for abbr in ["JST", "IST", "GMT"]:
    status, zones = resolve_abbreviation(abbr)
    print(f"\n{abbr}: {status}")
    for z in zones:
        print(f"    {z:<20} {snapshot(z)}")


JST: ambiguous
    Asia/Tokyo           17:24 JST (+0900)
    Japan                17:24 JST (+0900)

IST: ambiguous
    Asia/Calcutta        13:54 IST (+0530)
    Asia/Kolkata         13:54 IST (+0530)
    Eire                 09:24 IST (+0100)
    Europe/Dublin        09:24 IST (+0100)

GMT: ambiguous
    Africa/Abidjan       08:24 GMT (+0000)
    Africa/Accra         08:24 GMT (+0000)
    Africa/Bamako        08:24 GMT (+0000)
    Africa/Banjul        08:24 GMT (+0000)
    Africa/Bissau        08:24 GMT (+0000)
    Africa/Conakry       08:24 GMT (+0000)
    Africa/Dakar         08:24 GMT (+0000)
    Africa/Freetown      08:24 GMT (+0000)
    Africa/Lome          08:24 GMT (+0000)
    Africa/Monrovia      08:24 GMT (+0000)
    Africa/Nouakchott    08:24 GMT (+0000)
    Africa/Ouagadougou   08:24 GMT (+0000)
    Africa/Sao_Tome      08:24 GMT (+0000)
    Africa/Timbuktu      08:24 GMT (+0000)
    America/Danmarkshavn 08:24 GMT (+0000)
    Atlantic/Reykjavik   08:24 GMT (+0000)
    At